# Financial LLM Fine-Tuning: Qwen2.5-1.5B + LoRA + 4-bit (QLoRA)

**Run everything:** Runtime -> Change runtime type -> **T4 GPU**, then Runtime -> **Run all**.
The only manual step is pasting your Hugging Face **Write** token in the login cell.
No code edits needed: the notebook detects your username, trains, pushes the adapter, deploys the Space and builds the results bundle.

In [ ]:
!pip install -q -U transformers peft bitsandbytes trl datasets accelerate matplotlib

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
HF_USER = api.whoami()["name"]
ADAPTER_REPO = f"{HF_USER}/qwen2.5-1.5b-finance-lora"
SPACE_REPO = f"{HF_USER}/finance-llm"
print("User:", HF_USER)
print("Adapter repo:", ADAPTER_REPO)
print("Space repo:", SPACE_REPO)

In [ ]:
from datasets import load_dataset

SYSTEM = "You are a helpful financial assistant."
ds = load_dataset("gbharti/finance-alpaca", split="train").shuffle(seed=42).select(range(5000))

def to_chat(ex):
    user = ex["instruction"]
    if ex["input"]:
        user += "\n\n" + ex["input"]
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user},
        {"role": "assistant", "content": ex["output"]},
    ]}

ds = ds.map(to_chat, remove_columns=ds.column_names)
ds = ds.train_test_split(test_size=0.05, seed=42)
print(ds)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto"
)
mem_gb = model.get_memory_footprint() / 1e9
print(f"4-bit model memory footprint: {mem_gb:.2f} GB")

In [ ]:
from peft import LoraConfig

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
import time
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="qwen-fin-lora",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    max_length=512,   # older trl versions: max_seq_length=512
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    peft_config=peft_config,
    processing_class=tokenizer,   # older trl versions: tokenizer=tokenizer
)

trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"Trainable params: {trainable:,} of {total:,} ({100*trainable/total:.2f}%)")

# LoRA B matrices start at zero, so this equals the base model's eval loss
base_eval_loss = trainer.evaluate()["eval_loss"]
print("Base model eval loss:", round(base_eval_loss, 4))

t0 = time.time()
train_out = trainer.train()
train_minutes = (time.time() - t0) / 60

final_eval_loss = trainer.evaluate()["eval_loss"]
logs = trainer.state.log_history
final_train_loss = [l["loss"] for l in logs if "loss" in l][-1]
print(f"Train time: {train_minutes:.1f} min | final train loss {final_train_loss:.4f} | final eval loss {final_eval_loss:.4f}")

In [ ]:
import matplotlib.pyplot as plt

tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(7, 4))
plt.plot(*zip(*tr), label="train loss")
if ev:
    plt.plot(*zip(*ev), label="eval loss", marker="o")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Qwen2.5-1.5B finance QLoRA")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
prompts = [
    "What is the difference between stocks and bonds?",
    "How does compound interest work?",
    "What is a P/E ratio and why does it matter?",
]

def gen(prompt):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = trainer.model.generate(**inputs, max_new_tokens=200, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

trainer.model.eval()
comparisons = []
for p in prompts:
    tuned = gen(p)
    with trainer.model.disable_adapter():
        base = gen(p)
    comparisons.append({"prompt": p, "base": base, "tuned": tuned})
    print("Q:", p, "\n--- BASE ---\n", base[:300], "\n--- FINE-TUNED ---\n", tuned[:300], "\n")

In [ ]:
import os

trainer.model.save_pretrained("final_adapter")
adapter_mb = sum(os.path.getsize(os.path.join("final_adapter", f)) for f in os.listdir("final_adapter")) / 1e6
print(f"Adapter size: {adapter_mb:.1f} MB")

card = "\n".join([
    "---",
    "base_model: Qwen/Qwen2.5-1.5B-Instruct",
    "library_name: peft",
    "license: apache-2.0",
    "tags: [lora, qlora, finance, peft, trl, sft]",
    "---",
    "",
    "# Qwen2.5-1.5B Finance LoRA adapter",
    "",
    "QLoRA (4-bit NF4) fine-tune of Qwen2.5-1.5B-Instruct on a 5,000-example subset of gbharti/finance-alpaca.",
    "",
    f"- Eval loss: base {base_eval_loss:.4f} -> fine-tuned {final_eval_loss:.4f}",
    "- LoRA r=16, alpha=32, dropout=0.05, all attention and MLP projections",
    "- Educational use only. Not financial advice.",
])
open("final_adapter/README.md", "w").write(card)

api.create_repo(ADAPTER_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="final_adapter", repo_id=ADAPTER_REPO, repo_type="model")
api.upload_file(path_or_fileobj="loss_curve.png", path_in_repo="loss_curve.png", repo_id=ADAPTER_REPO, repo_type="model")
print("Adapter pushed: https://huggingface.co/" + ADAPTER_REPO)

In [ ]:
APP_SRC = r'''import os
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
_here = os.path.dirname(os.path.abspath(__file__))
ADAPTER = os.environ.get("ADAPTER_REPO") or open(os.path.join(_here, "adapter_repo.txt")).read().strip()
SYSTEM = "You are a helpful financial assistant."

tokenizer = AutoTokenizer.from_pretrained(BASE)
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
model = PeftModel.from_pretrained(model, ADAPTER)
model = model.merge_and_unload()
model.eval()


def answer(question):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.7)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


demo = gr.Interface(
    fn=answer,
    inputs=gr.Textbox(lines=4, label="Ask a finance question"),
    outputs=gr.Textbox(label="Answer"),
    title="Finance LLM (Qwen2.5-1.5B + LoRA, 4-bit QLoRA fine-tuned)",
    description="Educational demo only. Not financial advice. Running on free CPU, so answers take 20-60 seconds.",
    examples=[
        "What is the difference between stocks and bonds?",
        "How does compound interest work?",
        "What is a P/E ratio and why does it matter?",
    ],
)

if __name__ == "__main__":
    demo.launch()
'''
REQ_SRC = r'''torch
transformers
peft
accelerate
gradio
'''
SPACE_README = r'''---
title: Finance LLM
emoji: 💹
colorFrom: blue
colorTo: green
sdk: gradio
app_file: app.py
pinned: false
---
'''

api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)
for name, content in [("app.py", APP_SRC), ("requirements.txt", REQ_SRC), ("README.md", SPACE_README), ("adapter_repo.txt", ADAPTER_REPO)]:
    api.upload_file(path_or_fileobj=content.encode(), path_in_repo=name, repo_id=SPACE_REPO, repo_type="space")
print("Space deployed: https://huggingface.co/spaces/" + SPACE_REPO)
print("It takes 5-15 min to build the first time, then shows Running.")

In [ ]:
import json, shutil
from google.colab import files

os.makedirs("bundle", exist_ok=True)
metrics = {
    "base_model": model_id,
    "dataset": "gbharti/finance-alpaca (5000-sample subset, 95/5 split)",
    "base_eval_loss": round(base_eval_loss, 4),
    "final_eval_loss": round(final_eval_loss, 4),
    "final_train_loss": round(final_train_loss, 4),
    "trainable_params": int(trainable),
    "total_params": int(total),
    "adapter_size_mb": round(adapter_mb, 1),
    "model_memory_gb_4bit": round(mem_gb, 2),
    "train_minutes": round(train_minutes, 1),
}
json.dump(metrics, open("bundle/results.json", "w"), indent=2)
shutil.copy("loss_curve.png", "bundle/loss_curve.png")
open("bundle/adapter_repo.txt", "w").write(ADAPTER_REPO)
open("bundle/app.py", "w").write(APP_SRC)

L = []
L.append("# Financial LLM Fine-Tuning & Adaptation")
L.append("")
L.append("QLoRA fine-tuning of **Qwen2.5-1.5B-Instruct** on financial Q&A data using LoRA (PEFT), 4-bit NF4 quantization (BitsAndBytes) and TRL SFTTrainer, deployed as a Gradio app on Hugging Face Spaces.")
L.append("")
L.append("## Live links")
L.append(f"- Live demo: https://huggingface.co/spaces/{SPACE_REPO}")
L.append(f"- LoRA adapter: https://huggingface.co/{ADAPTER_REPO}")
L.append("")
L.append("## Results")
L.append("| Metric | Value |")
L.append("|---|---|")
L.append(f"| Eval loss (base model) | {base_eval_loss:.4f} |")
L.append(f"| Eval loss (fine-tuned) | {final_eval_loss:.4f} |")
L.append(f"| Final train loss | {final_train_loss:.4f} |")
L.append(f"| Trainable parameters | {trainable:,} ({100*trainable/total:.2f}%) |")
L.append(f"| Adapter size | {adapter_mb:.1f} MB |")
L.append(f"| 4-bit base model memory | {mem_gb:.2f} GB |")
L.append(f"| Training time (Colab T4) | {train_minutes:.1f} min |")
L.append("")
L.append("![loss curve](outputs/loss_curve.png)")
L.append("")
L.append("## Method")
L.append("| Item | Value |")
L.append("|---|---|")
L.append("| Base model | Qwen/Qwen2.5-1.5B-Instruct |")
L.append("| Dataset | gbharti/finance-alpaca, 5,000-sample subset, 95/5 train/eval split |")
L.append("| Quantization | 4-bit NF4, double quantization, fp16 compute |")
L.append("| LoRA | r=16, alpha=32, dropout=0.05, q/k/v/o/gate/up/down projections |")
L.append("| Optimizer | paged_adamw_8bit |")
L.append("| Memory optimizations | 4-bit weights, gradient checkpointing, gradient accumulation (effective batch 16) |")
L.append("| LR / schedule | 2e-4, cosine, 3% warmup, 1 epoch |")
L.append("")
L.append("## Base vs fine-tuned (greedy decoding)")
for c in comparisons:
    L.append("")
    L.append("**Q: " + c["prompt"] + "**")
    L.append("")
    L.append("Base:")
    L.append("> " + c["base"][:500].replace("\n", " "))
    L.append("")
    L.append("Fine-tuned:")
    L.append("> " + c["tuned"][:500].replace("\n", " "))
L.append("")
L.append("## Project structure")
L.append("```")
L.append("notebooks/finance_llm_finetune.ipynb   # train, push adapter, deploy Space, generate results")
L.append("app/                                   # Gradio app deployed to Hugging Face Spaces")
L.append("outputs/                               # loss curve and metrics")
L.append("```")
L.append("")
L.append("## Reproduce")
L.append("Open the notebook in Google Colab, select a T4 GPU, and Run all. Educational use only; not financial advice.")
open("bundle/README.md", "w").write("\n".join(L))

shutil.make_archive("project_results", "zip", "bundle")
files.download("project_results.zip")
print("Done. project_results.zip downloaded. Live demo: https://huggingface.co/spaces/" + SPACE_REPO)